# Short-squeeze session pattern — strategy backtest

Follow-up to [`discovery.ipynb`](./discovery.ipynb). The discovery study scored
days at session-aggregate level (4 legs, 10 checks). This notebook turns the
day-level pattern into a **tradeable bar-level signal** with tight risk
management, and backtests it.

## Signal

A bar-level long trigger fires when:

1. The bar is in London or NY session (UTC 07:00 - 21:00)
2. The day is a daily **short-macro day** (asia close < asia open, asia OI rose
   > 0.5%, asia mean funding < 0) — same gating used in `discovery.ipynb`
3. Bar low pierces the prior 6-hour low (sweep)
4. Bar's perp_cvd is strongly negative (< -100) AND bar's spot_cvd is much
   less negative — divergence gap > +200. This is the "bullish divergence
   between perp and spot CVD" cue from the original trader description.
5. Close in upper portion of bar (close_in_range ≥ 0.25) — must have some
   wick reversal, not closing at the low.

Mirror for shorts (sweep high + bearish divergence + close in lower portion).

Cooldown: 4h per side. Resolution: 15m bars.

## Execution

- Entry at the close of the trigger bar (next 15m bar's open in practice)
- Stop at `trigger_bar.low * 0.999` (long) or `trigger_bar.high * 1.001` (short)
- Targets tested: 1.0R / 1.5R / 2.0R / 3.0R
- Time stop: 6 hours from entry
- Stops/targets walked on `btc_1m` (spot 1m bars)
- Slippage: 2 bp on each leg

## Data dependencies

Requires the 15m kline tables added on 2026-05-18:
- `cd_futures_15m` — Binance perp 15m with buy/sell volume split
- `cd_spot_15m`    — Binance spot 15m with buy/sell volume split

Both are populated by `data/sources/binance.py --backfill-klines-15m` and
self-maintained by the live feed's `refresh_all()` and `fix_all_gaps()`.

## Headline result — long side @ 3.0R TP (4 yrs)

`SIGNAL_PARAMS`: **percentile thresholds** — `perp_cvd_pct < 0.15`,
`divergence_pct > 0.70`, `lookback=24`, `cir=0.10`. Switched from absolute
(`perp_cvd<-400`, `divergence>200`) to percentile on 2026-05-18 for volume-
drift robustness. Sensitivity analysis showed the divergence threshold is
non-binding within [0.70, 0.80] (perp and divergence are highly correlated,
so the perp filter does all the work), so we use the looser 0.70 for forward
robustness without changing any historical result.

| Side | n | Win% | Avg R | PF |
|---|---:|---:|---:|---:|
| **long (percentile, current)** | **70** | 44.3% | **+0.40** | **1.65** |
| long (absolute, prior baseline) | 71 | 46.5% | +0.49 | 1.83 |
| short | (does not work at any configuration) | — | — | — |

The percentile version has ~3× **lower year-to-year variance** at a cost of
~20% lower headline avg R. It's the right choice for live deployment because
it auto-adapts as market structure evolves.

Only the long side has an edge. The short side (fading longs at a sweep high)
consistently loses, despite the long-squeeze *daily pattern* being 4.6× more
common than the short-squeeze daily pattern. That asymmetry is the most
interesting structural finding here.

In [ ]:
from __future__ import annotations

import datetime as dt
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('could not locate prod.db walking up from cwd')
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'databases' / 'prod.db'

SESSIONS = {'asia': (0, 7), 'london': (7, 14), 'ny': (14, 21)}
LOOKBACK_BARS = 24      # 6 hours at 15m
COOLDOWN_BARS = 16      # 4 hours
TIME_STOP_HOURS = 6
SLIPPAGE_BP = 2

print(f'DB: {DB}')

## Load 15m perp + spot, plus hourly OI + funding for the macro filter

In [ ]:
def _load(table: str) -> pd.DataFrame:
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(f'SELECT * FROM {table} ORDER BY timestamp', con)
    con.close()
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df = df.set_index('ts')
    return df[~df.index.duplicated(keep='last')]

perp15 = _load('cd_futures_15m')
spot15 = _load('cd_spot_15m')
perp_h = _load('cd_futures_ohlcv')
oi_h   = _load('cd_open_interest')
fund_h = _load('cd_funding_rate')

start = oi_h.index.min()
perp15 = perp15.loc[start:]
spot15 = spot15.loc[start:]
perp_h = perp_h.loc[start:]
print(f'15m perp {len(perp15):,}   15m spot {len(spot15):,}   hourly perp {len(perp_h):,}')

# btc_1m for execution
con = sqlite3.connect(str(DB))
m1 = pd.read_sql(
    'SELECT open_time, open, high, low, close FROM btc_1m WHERE open_time >= ? ORDER BY open_time',
    con, params=(int(start.timestamp() * 1000),)
)
con.close()
m1['ts'] = pd.to_datetime(m1['open_time'], unit='ms', utc=True)
m1 = m1.set_index('ts').drop(columns='open_time')
m1 = m1[~m1.index.duplicated(keep='last')]
print(f'btc_1m {len(m1):,}')

## Build the 15m enriched frame

In [ ]:
b15 = pd.DataFrame(index=perp15.index)
b15['o']  = perp15['open']; b15['h'] = perp15['high']
b15['l']  = perp15['low'];  b15['c'] = perp15['close']
b15['perp_vol'] = perp15['volume']
b15['perp_cvd'] = perp15['volume_buy'] - perp15['volume_sell']
b15['spot_cvd'] = (spot15['volume_buy'] - spot15['volume_sell']).reindex(b15.index)
b15 = b15.dropna(subset=['c', 'perp_cvd', 'spot_cvd']).copy()

def _session_of(h):
    for name, (lo, hi) in SESSIONS.items():
        if lo <= h < hi:
            return name
    return None

b15['session'] = b15.index.hour.map(_session_of)
b15 = b15.dropna(subset=['session']).copy()
b15['date'] = b15.index.date

b15['prior_low_24']  = b15['l'].rolling(LOOKBACK_BARS + 1).min().shift(1)
b15['prior_high_24'] = b15['h'].rolling(LOOKBACK_BARS + 1).max().shift(1)

bar_range = (b15['h'] - b15['l']).clip(lower=1e-9)
b15['close_in_range'] = (b15['c'] - b15['l']) / bar_range

print(f'{len(b15):,} usable 15m bars from {b15.index.min()} to {b15.index.max()}')

## Rolling-percentile features (added 2026-05-18)

For volume drift robustness, we compute each bar's percentile rank within
the **trailing 90 days of London/NY bars**:

- `perp_cvd_pct` — what fraction of recent bars had a perp_cvd *less than or
  equal to* this one. Small values = bar is in the very negative tail
  (heavy short-side aggression).
- `divergence_pct` — same idea for `spot_cvd - perp_cvd`. Large values =
  divergence is in the positive tail.

A 90-day London/NY universe is ~5,000 bars — plenty for stable percentile
estimation. Computation runs in ~2 seconds.

In [ ]:
def rolling_percentile(series: pd.Series, window: int) -> pd.Series:
    """Trailing-window percentile rank (does not include the current value
    in its own distribution — i.e., today's bar is ranked against the prior
    `window` bars only)."""
    out = np.full(len(series), np.nan)
    arr = series.values
    for i in range(window, len(arr)):
        out[i] = (arr[i - window: i] <= arr[i]).mean()
    return pd.Series(out, index=series.index)

b15['divergence'] = b15['spot_cvd'] - b15['perp_cvd']

# Universe: all London/NY bars, with ~5000 bars (~90 days) trailing window.
WINDOW_BARS = 90 * 56
univ = b15[b15['session'].isin(['london', 'ny'])].copy()
univ['perp_cvd_pct']   = rolling_percentile(univ['perp_cvd'], WINDOW_BARS)
univ['divergence_pct'] = rolling_percentile(univ['divergence'], WINDOW_BARS)

b15['perp_cvd_pct']   = univ['perp_cvd_pct'].reindex(b15.index)
b15['divergence_pct'] = univ['divergence_pct'].reindex(b15.index)

# Diagnostic: how do the legacy absolute thresholds map into this percentile space?
in_lonny = b15['session'].isin(['london', 'ny'])
abs_perp_pct = (b15.loc[in_lonny, 'perp_cvd']   < -400).mean()
abs_div_pct  = (b15.loc[in_lonny, 'divergence'] >  200).mean()
print(f'absolute thresholds map to:')
print(f'  perp_cvd < -400   ->  bottom {abs_perp_pct:.1%} of bars')
print(f'  divergence > 200  ->  top    {abs_div_pct:.1%} of bars')
print(f'\nlive thresholds are set in the trigger cell below (perp_pct < 0.15, div_pct > 0.70)')

## Daily macro context

In [ ]:
def _agg_asia(g):
    asia = g[g.index.hour < 7]
    if len(asia) < 2:
        return pd.Series({'close_lt_open': False, 'close_gt_open': False,
                          'oi_pct': 0.0, 'fund_mean': 0.0})
    oi_series = oi_h['oi_close'].reindex(asia.index).ffill(limit=3)
    oi_o, oi_c = oi_series.iloc[0], oi_series.iloc[-1]
    f = fund_h['fr_close'].reindex(asia.index).ffill(limit=8).mean()
    oi_pct = 0.0 if (pd.isna(oi_o) or pd.isna(oi_c)) else (oi_c - oi_o) / max(oi_o, 1e-9)
    return pd.Series({
        'close_lt_open': asia['close'].iloc[-1] < asia['open'].iloc[0],
        'close_gt_open': asia['close'].iloc[-1] > asia['open'].iloc[0],
        'oi_pct': oi_pct,
        'fund_mean': f if pd.notna(f) else 0.0,
    })

perp_h_d = perp_h.copy()
perp_h_d['date'] = perp_h_d.index.date
day_ctx = perp_h_d.groupby('date').apply(_agg_asia, include_groups=False)
if isinstance(day_ctx.index, pd.MultiIndex):
    day_ctx = day_ctx.unstack(-1)
day_ctx['short_macro'] = day_ctx['close_lt_open'] & (day_ctx['oi_pct'] > 0.005) & (day_ctx['fund_mean'] < 0)
day_ctx['long_macro']  = day_ctx['close_gt_open'] & (day_ctx['oi_pct'] > 0.005) & (day_ctx['fund_mean'] > 0)

b15['short_macro'] = b15['date'].map(day_ctx['short_macro'].to_dict()).fillna(False).astype(bool)
b15['long_macro']  = b15['date'].map(day_ctx['long_macro'].to_dict()).fillna(False).astype(bool)
print(f'short macro days: {int(day_ctx["short_macro"].sum())}')
print(f'long  macro days: {int(day_ctx["long_macro"].sum())}')

## Trigger detection — bar-level signal

In [ ]:
SIGNAL_PARAMS = {
    # Percentile thresholds (live deployment). Sensitivity analysis 2026-05-18
    # showed: (a) within div_pct ∈ [0.70, 0.80] the divergence threshold is
    # non-binding because every bar in the bottom 15% of perp_cvd is already
    # in the top 30% of divergence (correlated); (b) loosening div_pct to
    # 0.70 produces identical historical results AND is more robust if the
    # perp/spot CVD correlation drifts in the future.
    'perp_cvd_pct_max':       0.15,
    'divergence_pct_min':     0.70,
    'close_in_range_min':     0.10,
    'close_in_range_max':     0.90,    # short side (kept for completeness)
    # Legacy absolute thresholds — used by the threshold-sweep section below
    # for the calibration story; not used by the live trigger.
    'perp_cvd_threshold':     400.0,
    'divergence_gap':         200.0,
}

in_window = b15['session'].isin(['london', 'ny'])

# Percentile-threshold trigger (live deployment).
long_trigger = (
    in_window
    & b15['short_macro']
    & (b15['l'] < b15['prior_low_24'])
    & (b15['perp_cvd_pct']   < SIGNAL_PARAMS['perp_cvd_pct_max'])
    & (b15['divergence_pct'] > SIGNAL_PARAMS['divergence_pct_min'])
    & (b15['close_in_range'] >= SIGNAL_PARAMS['close_in_range_min'])
)

# Short side trigger — kept for completeness but does not produce a positive
# edge at any TP configuration; not used live. Uses percentile thresholds on
# the opposite tails: top X% of perp_cvd, bottom Y% of divergence.
short_trigger = (
    in_window
    & b15['long_macro']
    & (b15['h'] > b15['prior_high_24'])
    & (b15['perp_cvd_pct']   > 1 - SIGNAL_PARAMS['perp_cvd_pct_max'])
    & (b15['divergence_pct'] < 1 - SIGNAL_PARAMS['divergence_pct_min'])
    & (b15['close_in_range'] <= SIGNAL_PARAMS['close_in_range_max'])
)

def _cooldown(mask: pd.Series, bars: int = COOLDOWN_BARS) -> pd.Series:
    out = pd.Series(False, index=mask.index)
    last = None
    for ts, v in mask.items():
        if not v:
            continue
        if last is None or (ts - last) >= pd.Timedelta(minutes=15 * bars):
            out.loc[ts] = True
            last = ts
    return out

long_trigger  = _cooldown(long_trigger)
short_trigger = _cooldown(short_trigger)
print(f'long triggers  (pre-cooldown was higher, this is final): {int(long_trigger.sum())}')
print(f'short triggers: {int(short_trigger.sum())}')

## Execution simulator

In [ ]:
def simulate(trigger_ts: pd.Timestamp, direction: str, tp_R: float) -> dict | None:
    bar = b15.loc[trigger_ts]
    entry = bar['c']
    if direction == 'long':
        stop = bar['l'] * (1 - 0.001)
        risk = entry - stop
        if risk <= 0: return None
        target = entry + tp_R * risk
    else:
        stop = bar['h'] * (1 + 0.001)
        risk = stop - entry
        if risk <= 0: return None
        target = entry - tp_R * risk

    t0 = trigger_ts + pd.Timedelta(minutes=15)
    t_end = t0 + pd.Timedelta(hours=TIME_STOP_HOURS)
    path = m1.loc[t0:t_end]
    if path.empty: return None

    exit_price = None; exit_reason = None; exit_ts = None
    for ts, b1 in path.iterrows():
        if direction == 'long':
            if b1['low']  <= stop:   exit_price, exit_reason, exit_ts = stop,   'stop',   ts; break
            if b1['high'] >= target: exit_price, exit_reason, exit_ts = target, 'target', ts; break
        else:
            if b1['high'] >= stop:   exit_price, exit_reason, exit_ts = stop,   'stop',   ts; break
            if b1['low']  <= target: exit_price, exit_reason, exit_ts = target, 'target', ts; break
    if exit_price is None:
        exit_price = path.iloc[-1]['close']
        exit_reason = 'time'
        exit_ts = path.index[-1]

    slip = SLIPPAGE_BP / 1e4
    if direction == 'long':
        eff_e = entry * (1 + slip); eff_x = exit_price * (1 - slip)
        pnl_r   = (eff_x - eff_e) / risk
        pnl_pct = (eff_x - eff_e) / eff_e
    else:
        eff_e = entry * (1 - slip); eff_x = exit_price * (1 + slip)
        pnl_r   = (eff_e - eff_x) / risk
        pnl_pct = (eff_e - eff_x) / eff_e

    return {
        'trigger_ts': trigger_ts, 'direction': direction, 'tp_R': tp_R,
        'entry': entry, 'stop': stop, 'target': target,
        'exit_price': exit_price, 'exit_reason': exit_reason, 'exit_ts': exit_ts,
        'hold_min': (exit_ts - trigger_ts).total_seconds() / 60,
        'pnl_R': pnl_r, 'pnl_pct': pnl_pct,
        'risk_pct': risk / entry,
    }

def run_set(mask: pd.Series, direction: str, tp_R: float) -> pd.DataFrame:
    rows = []
    for ts in mask.index[mask]:
        r = simulate(ts, direction, tp_R)
        if r is not None:
            rows.append(r)
    return pd.DataFrame(rows)

def stats_for(df: pd.DataFrame) -> dict:
    if len(df) == 0:
        return {'n': 0}
    wins   = df.loc[df['pnl_R'] > 0, 'pnl_R'].sum()
    losses = -df.loc[df['pnl_R'] < 0, 'pnl_R'].sum()
    pf = wins / losses if losses > 0 else float('inf')
    return {
        'n':       len(df),
        'win_pct': (df['pnl_R'] > 0).mean(),
        'avg_R':   df['pnl_R'].mean(),
        'med_R':   df['pnl_R'].median(),
        'PF':      pf,
        'E[pct]':  df['pnl_pct'].mean() * 100,
        'avg_hold_min': df['hold_min'].mean(),
    }

## Run across TP multipliers

In [ ]:
tps = [1.0, 1.5, 2.0, 3.0]
rows = []
results = {}
for side, trig in [('long', long_trigger), ('short', short_trigger)]:
    for tp in tps:
        df = run_set(trig, side, tp)
        results[(side, tp)] = df
        s = stats_for(df)
        s['side'] = side
        s['tp_R'] = tp
        rows.append(s)

summary = pd.DataFrame(rows)
summary = summary[['side','tp_R','n','win_pct','avg_R','med_R','PF','E[pct]','avg_hold_min']]
summary.round(3)

## Year-by-year breakdown for the best TP per side

In [ ]:
def year_table(df: pd.DataFrame, label: str):
    if df is None or len(df) == 0:
        print(f'{label}: no trades'); return None
    d = df.copy()
    d['year'] = pd.to_datetime(d['trigger_ts']).dt.year
    out = d.groupby('year').agg(
        n=('pnl_R', 'size'),
        win_pct=('pnl_R', lambda x: (x > 0).mean()),
        avg_R=('pnl_R', 'mean'),
        sum_R=('pnl_R', 'sum'),
        avg_pct=('pnl_pct', 'mean'),
    ).round(3)
    print(f'\n--- {label} ---')
    print(out.to_string())
    return out

# Pick best TP per side by avg R
def _best_tp(side):
    best = None; best_avg = -1e9
    for tp in tps:
        df = results[(side, tp)]
        if len(df) == 0: continue
        a = df['pnl_R'].mean()
        if a > best_avg:
            best_avg = a; best = tp
    return best

best_long  = _best_tp('long')
best_short = _best_tp('short')
print(f'best long TP:  {best_long}')
print(f'best short TP: {best_short}')

year_table(results[('long', best_long)],  f'LONG @ {best_long}R')
if best_short is not None:
    year_table(results[('short', best_short)], f'SHORT @ {best_short}R')

## Equity curve — best long setup

In [ ]:
plt.style.use('dark_background')

def equity_curve(df: pd.DataFrame, label: str, risk_pct: float = 0.01):
    if len(df) == 0:
        print(f'{label}: no trades'); return
    d = df.sort_values('trigger_ts').copy()
    d['equity_R']   = d['pnl_R'].cumsum()
    d['equity_pct'] = (1 + d['pnl_R'] * risk_pct).cumprod() - 1
    final_R   = d['equity_R'].iloc[-1]
    final_pct = d['equity_pct'].iloc[-1] * 100
    # Max drawdown in R
    peak = d['equity_R'].cummax()
    dd = (d['equity_R'] - peak)
    max_dd_R = dd.min()

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                              gridspec_kw={'height_ratios': [2.5, 1]})
    axes[0].plot(d['trigger_ts'], d['equity_R'], color='cyan', lw=1.2)
    axes[0].fill_between(d['trigger_ts'], d['equity_R'], 0,
                         where=(d['equity_R'] >= 0), color='green', alpha=0.10)
    axes[0].fill_between(d['trigger_ts'], d['equity_R'], 0,
                         where=(d['equity_R'] < 0),  color='red', alpha=0.10)
    axes[0].set_title(
        f'{label}  •  cumulative R   '
        f'(final {final_R:+.1f}R, max DD {max_dd_R:.1f}R, '
        f'@{risk_pct*100:.1f}% risk -> {final_pct:+.1f}% NAV)'
    )
    axes[0].set_ylabel('cumulative R'); axes[0].axhline(0, color='gray', lw=0.5)

    axes[1].bar(d['trigger_ts'], d['pnl_R'],
                color=np.where(d['pnl_R'] > 0, 'green', 'red'),
                width=2, alpha=0.6)
    axes[1].axhline(0, color='gray', lw=0.5)
    axes[1].set_ylabel('per-trade R')

    plt.tight_layout(); plt.show()

equity_curve(results[('long', best_long)], f'LONG @ {best_long}R')

## Case study — 2026-02-11 (the screenshot day)

In [ ]:
target = dt.date(2026, 2, 11)
day_long = long_trigger[long_trigger.index.date == target]
print(f'long triggers on {target}: {int(day_long.sum())}')

for ts in day_long[day_long].index:
    bar = b15.loc[ts]
    res = simulate(ts, 'long', best_long)
    print(f'\n{ts}')
    print(f'  bar: low={bar.l:.0f} close={bar.c:.0f}  perp_cvd={bar.perp_cvd:+.0f}  spot_cvd={bar.spot_cvd:+.0f}  divergence={bar.spot_cvd - bar.perp_cvd:+.0f}')
    if res:
        print(f'  entry={res["entry"]:.0f}  stop={res["stop"]:.0f}  target({best_long}R)={res["target"]:.0f}')
        print(f'  exit {res["exit_price"]:.0f} via {res["exit_reason"]} at {res["exit_ts"]}  -> {res["pnl_R"]:+.2f}R ({res["pnl_pct"]*100:+.2f}%) in {res["hold_min"]:.0f}m')

## Distribution of per-trade outcomes

In [ ]:
df_long = results[('long', best_long)]
if len(df_long) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_long['pnl_R'], bins=30, color='cyan', alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='gray', lw=0.8)
    ax.axvline(df_long['pnl_R'].mean(), color='yellow', lw=1.2, label=f'mean {df_long["pnl_R"].mean():+.2f}R')
    ax.axvline(df_long['pnl_R'].median(), color='orange', lw=1.2, ls='--', label=f'median {df_long["pnl_R"].median():+.2f}R')
    ax.legend(); ax.set_xlabel('per-trade R')
    ax.set_title(f'LONG @ {best_long}R — distribution of per-trade returns (n={len(df_long)})')
    plt.tight_layout(); plt.show()

## Exit rule study — fixed TP vs trailing variants

The leverage-scalper instinct is to use a tight trailing stop on this kind of
signal. Counterintuitive finding: **every trailing variant tested here
underperforms the fixed 3R target**.

Why: this is a *mean-reversion* signal, not a runaway trend. Post-trigger paths
resolve to ~1-3R targets and then chop. Tight trails get shaken out by normal
mid-trade pullbacks on the way to the level, eating the edge.

Six exit rules compared on the same 60 long triggers:

| ID | Rule |
|---|---|
| V1 | Fixed 3R (baseline) |
| V2 | Move SL to BE after +1R, keep 3R target |
| V3 | Step-lock: BE@1R, +1R@2R, +2R@3R |
| V4 | Tight 1m chandelier (5-bar low) after +1R, no fixed target |
| V5 | Loose 1m chandelier (15-bar low) after +1R, no fixed target |
| V6 | 0.4% fixed-% trail after +1R, no fixed target |

In [ ]:
# State-machine exit-rule simulator. Walks btc_1m bar-by-bar; the exit_rule
# closure inspects/mutates `state` and returns either an exit action or None.

def walk_long(trigger_ts, exit_rule):
    bar = b15.loc[trigger_ts]
    entry = bar['c']
    init_stop = bar['l'] * (1 - 0.001)
    risk = entry - init_stop
    if risk <= 0:
        return None
    state = {
        'entry': entry, 'init_stop': init_stop, 'init_target': entry + 3 * risk,
        'risk': risk, 'current_stop': init_stop, 'running_max': entry,
        'running_max_R': 0.0, 'hit_1R': False, 'hit_2R': False,
    }
    t0 = trigger_ts + pd.Timedelta(minutes=15)
    t_end = t0 + pd.Timedelta(hours=TIME_STOP_HOURS)
    path = m1.loc[t0:t_end]
    if path.empty: return None

    for ts, b1 in path.iterrows():
        if b1['high'] > state['running_max']:
            state['running_max']   = b1['high']
            state['running_max_R'] = (b1['high'] - entry) / risk
            if state['running_max_R'] >= 1.0: state['hit_1R'] = True
            if state['running_max_R'] >= 2.0: state['hit_2R'] = True
        action, px = exit_rule(state, b1, ts, path)
        if action is not None:
            return {'exit_reason': action, 'exit_price': px, 'exit_ts': ts, 'state': state}
    return {'exit_reason': 'time', 'exit_price': path.iloc[-1]['close'],
            'exit_ts': path.index[-1], 'state': state}

# Exit rules ------------------------------------------------------------------

def v1_fixed_3R(state, b1, ts, path):
    if b1['low']  <= state['current_stop']: return 'stop',   state['current_stop']
    if b1['high'] >= state['init_target']:  return 'target', state['init_target']
    return None, None

def v2_be_at_1R(state, b1, ts, path):
    if state['hit_1R']:
        state['current_stop'] = max(state['current_stop'], state['entry'])
    if b1['low']  <= state['current_stop']: return 'stop',   state['current_stop']
    if b1['high'] >= state['init_target']:  return 'target', state['init_target']
    return None, None

def v3_step_lock(state, b1, ts, path):
    if state['hit_1R']:
        state['current_stop'] = max(state['current_stop'], state['entry'])
    if state['hit_2R']:
        state['current_stop'] = max(state['current_stop'], state['entry'] + state['risk'])
    if b1['low']  <= state['current_stop']: return 'stop',   state['current_stop']
    if b1['high'] >= state['init_target']:  return 'target', state['init_target']
    return None, None

def _chandelier(state, b1, ts, path, lookback_bars):
    if state['hit_1R']:
        loc = path.index.get_loc(ts)
        if loc >= lookback_bars:
            recent_low = path.iloc[loc - lookback_bars: loc]['low'].min()
            state['current_stop'] = max(state['current_stop'], recent_low * 0.999)
    if b1['low'] <= state['current_stop']:
        return ('trail' if state['hit_1R'] else 'stop'), state['current_stop']
    return None, None

def v4_tight_chandelier(state, b1, ts, path):  return _chandelier(state, b1, ts, path, 5)
def v5_loose_chandelier(state, b1, ts, path):  return _chandelier(state, b1, ts, path, 15)

def v6_pct_trail(state, b1, ts, path):
    if state['hit_1R']:
        trail = state['running_max'] * (1 - 0.004)
        state['current_stop'] = max(state['current_stop'], trail)
    if b1['low'] <= state['current_stop']:
        return ('trail' if state['hit_1R'] else 'stop'), state['current_stop']
    return None, None

VARIANTS = {
    'V1 fixed 3R':            v1_fixed_3R,
    'V2 BE@1R + 3R tp':       v2_be_at_1R,
    'V3 step-lock':           v3_step_lock,
    'V4 tight 5m chand.':     v4_tight_chandelier,
    'V5 loose 15m chand.':    v5_loose_chandelier,
    'V6 0.4% pct trail':      v6_pct_trail,
}

def run_variant(rule):
    rows = []
    slip = SLIPPAGE_BP / 1e4
    for ts in long_trigger.index[long_trigger]:
        r = walk_long(ts, rule)
        if r is None: continue
        entry = r['state']['entry']; risk = r['state']['risk']
        eff_e = entry * (1 + slip); eff_x = r['exit_price'] * (1 - slip)
        pnl_r = (eff_x - eff_e) / risk
        pnl_pct = (eff_x - eff_e) / eff_e
        rows.append({
            'trigger_ts': ts, 'exit_reason': r['exit_reason'],
            'exit_ts': r['exit_ts'], 'pnl_R': pnl_r, 'pnl_pct': pnl_pct,
            'hold_min': (r['exit_ts'] - ts).total_seconds() / 60,
            'running_max_R': r['state']['running_max_R'],
        })
    return pd.DataFrame(rows)

tsl_results = {}
tsl_summary_rows = []
for label, rule in VARIANTS.items():
    df = run_variant(rule)
    tsl_results[label] = df
    if len(df) == 0: continue
    eq = df.sort_values('trigger_ts')['pnl_R'].cumsum()
    dd = (eq - eq.cummax()).min()
    wins   = df.loc[df['pnl_R'] > 0, 'pnl_R'].sum()
    losses = -df.loc[df['pnl_R'] < 0, 'pnl_R'].sum()
    pf = wins / losses if losses > 0 else float('inf')
    tsl_summary_rows.append({
        'variant':    label,
        'n':          len(df),
        'win_pct':    (df['pnl_R'] > 0).mean(),
        'avg_R':      df['pnl_R'].mean(),
        'med_R':      df['pnl_R'].median(),
        'sum_R':      df['pnl_R'].sum(),
        'PF':         pf,
        'max_dd_R':   dd,
        'E[pct]':     df['pnl_pct'].mean() * 100,
        'avg_hold_m': df['hold_min'].mean(),
    })

tsl_summary = pd.DataFrame(tsl_summary_rows).set_index('variant')
tsl_summary.round(3)

### Cumulative R per variant — visual comparison

In [ ]:
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(11, 5))
for label, df in tsl_results.items():
    if len(df) == 0: continue
    d = df.sort_values('trigger_ts')
    eq = d['pnl_R'].cumsum()
    ax.plot(d['trigger_ts'], eq, label=label, lw=1.3)
ax.axhline(0, color='gray', lw=0.5)
ax.set_title('Cumulative R per exit rule (same 60 long triggers)')
ax.set_ylabel('cumulative R')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

### Running-max distribution — why trailing fails here

Across the 60 long triggers, here's how far the trade's running maximum got
before either stopping out or time-stopping. Most trades top out at +1-3R;
the long-tail extends to +5R but very few reach further. That distribution
is incompatible with tight TSLs.

In [ ]:
mxR = tsl_results['V1 fixed 3R']['running_max_R']
print('per-trigger maximum unrealized profit (R):')
print(mxR.describe().round(2).to_string())
print('\nrunning_max_R exceeded each level:')
for r in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    n = int((mxR >= r).sum())
    print(f'  >= {r:4.1f}R:  {n:3d}/{len(mxR)}  ({n/len(mxR):.1%})')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mxR, bins=30, color='lime', alpha=0.7, edgecolor='white', linewidth=0.5)
ax.axvline(1, color='gray', lw=0.8, label='1R')
ax.axvline(3, color='yellow', lw=1.2, label='3R target')
ax.set_xlabel('running max unrealized profit (R)')
ax.set_title('Distribution of peak unrealized R per trade')
ax.legend(); plt.tight_layout(); plt.show()

### Exit-rule conclusion

Fixed 3R wins by a wide margin. The implication for high-leverage deployment:
**don't trail.** Use a fixed entry / fixed stop / fixed target. The
counterintuitive part — that TSL underperforms here — is mechanically tied to
the signal type (mean-reversion to a level, not trend continuation).

The Feb 11 NY trade is the textbook example: ran to +3.85R unrealized max but
took 240 minutes to do so with several mid-trade pullbacks. Fixed 3R captured
+2.94R; the tight chandelier exited at -0.19R after a normal retrace; the
0.4% trail exited at +0.39R. **6-7× difference** in realized R on the same
underlying move, driven entirely by the exit rule.

## Threshold grid sweep

`SIGNAL_PARAMS` at the trigger cell was picked by inspection. Sweep four
parameters to see if the edge holds across nearby values (robust) or
collapses (overfit). Parameters:

- `divergence_gap` — min spot_cvd − perp_cvd at the trigger bar
- `close_in_range_min` — bar's close position within its range (reversal strength)
- `perp_cvd_threshold` — min |perp_cvd| at the trigger bar
- `lookback_bars` — sweep-detection window (24 = 6h, 48 = 12h, …)

For speed we precompute one execution per **candidate bar** (any bar that
passes the structural sweep + macro + session filters) at the chosen exit
rule (V1 fixed 3R), then aggregate stats per parameter combo by filtering.

*This cell takes ~30-60 seconds.*

In [ ]:
def candidate_bars(lookback_bars: int) -> pd.DataFrame:
    """Return all bars passing the structural test (sweep + macro + session)
    at this lookback, with the feature columns needed for threshold filtering."""
    prior_low = b15['l'].rolling(lookback_bars + 1).min().shift(1)
    candidates = b15[(b15['session'].isin(['london', 'ny']))
                      & (b15['short_macro'])
                      & (b15['l'] < prior_low)].copy()
    return candidates

slip = SLIPPAGE_BP / 1e4
def precompute_executions(candidates: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ts, bar in candidates.iterrows():
        r = walk_long(ts, v1_fixed_3R)
        if r is None: continue
        entry = r['state']['entry']; risk = r['state']['risk']
        eff_e = entry * (1 + slip); eff_x = r['exit_price'] * (1 - slip)
        pnl_r = (eff_x - eff_e) / risk
        rows.append({
            'trigger_ts': ts,
            'perp_cvd': bar['perp_cvd'],
            'spot_cvd': bar['spot_cvd'],
            'divergence': bar['spot_cvd'] - bar['perp_cvd'],
            'close_in_range': bar['close_in_range'],
            'pnl_R': pnl_r,
            'exit_reason': r['exit_reason'],
        })
    return pd.DataFrame(rows)

LOOKBACKS = [12, 24, 48, 96]   # 3h, 6h, 12h, 24h
DIV_GAPS  = [50, 100, 200, 400, 800]
CIR_MINS  = [0.10, 0.25, 0.40, 0.55, 0.70]
PERP_THRS = [50, 100, 200, 400, 800]

precomputed = {}
for lb in LOOKBACKS:
    cands = candidate_bars(lb)
    precomputed[lb] = precompute_executions(cands)
    print(f'  lookback={lb:3d}: {len(cands):4d} candidate bars, {len(precomputed[lb]):4d} executable')

In [ ]:
# Apply cooldown to a list of trigger timestamps
def cooldown_select(ts_series: pd.Series, bars: int = COOLDOWN_BARS) -> pd.Series:
    keep = []
    last = None
    for ts in ts_series:
        if last is None or (ts - last) >= pd.Timedelta(minutes=15 * bars):
            keep.append(ts); last = ts
    return pd.Series(keep, dtype='datetime64[ns, UTC]')

sweep_rows = []
for lb in LOOKBACKS:
    pre = precomputed[lb]
    for dg in DIV_GAPS:
        for cir in CIR_MINS:
            for pt in PERP_THRS:
                f = pre[(pre['perp_cvd'] < -pt)
                       & (pre['divergence'] > dg)
                       & (pre['close_in_range'] >= cir)]
                if len(f) == 0: continue
                keep_ts = cooldown_select(f['trigger_ts'])
                ff = f[f['trigger_ts'].isin(keep_ts)]
                if len(ff) < 5: continue   # require minimum sample
                wins = ff.loc[ff['pnl_R'] > 0, 'pnl_R'].sum()
                losses = -ff.loc[ff['pnl_R'] < 0, 'pnl_R'].sum()
                pf = wins / losses if losses > 0 else float('inf')
                sweep_rows.append({
                    'lookback': lb, 'div_gap': dg, 'cir': cir, 'perp_thr': pt,
                    'n':        len(ff),
                    'win_pct':  (ff['pnl_R'] > 0).mean(),
                    'avg_R':    ff['pnl_R'].mean(),
                    'sum_R':    ff['pnl_R'].sum(),
                    'PF':       pf,
                })

sweep = pd.DataFrame(sweep_rows)
print(f'{len(sweep)} parameter combos with n>=5')

### Top combos by avg R (requiring n ≥ 30 for sample-size sanity)

The Goldilocks point is high avg R *and* enough trades that the result isn't
luck. Show top 20 combos with at least 30 trades.

In [ ]:
robust = sweep[sweep['n'] >= 30].sort_values('avg_R', ascending=False).head(20)
robust.round(3)

### Sensitivity heatmap

Fix `lookback=24` (the baseline) and `perp_thr=100`, then show avg R as a
2D heatmap over `div_gap × close_in_range`. Look for a smooth high-avg-R
plateau (robust) vs an isolated spike (overfit).

In [ ]:
fixed_lb = 24
fixed_pt = 100
sub = sweep[(sweep['lookback'] == fixed_lb) & (sweep['perp_thr'] == fixed_pt)].copy()
piv_R = sub.pivot(index='cir', columns='div_gap', values='avg_R')
piv_n = sub.pivot(index='cir', columns='div_gap', values='n')

fig, (axR, axN) = plt.subplots(1, 2, figsize=(13, 4.5))
imR = axR.imshow(piv_R.values, aspect='auto', cmap='RdYlGn',
                 vmin=-0.5, vmax=0.5, origin='lower')
axR.set_xticks(range(len(piv_R.columns))); axR.set_xticklabels(piv_R.columns)
axR.set_yticks(range(len(piv_R.index)));   axR.set_yticklabels(piv_R.index)
axR.set_xlabel('div_gap'); axR.set_ylabel('close_in_range_min')
axR.set_title(f'avg R  (lookback={fixed_lb}, perp_thr={fixed_pt})')
for i in range(piv_R.shape[0]):
    for j in range(piv_R.shape[1]):
        v = piv_R.values[i, j]
        if pd.notna(v):
            axR.text(j, i, f'{v:+.2f}', ha='center', va='center',
                     color='black' if abs(v) < 0.3 else 'white', fontsize=9)
plt.colorbar(imR, ax=axR, fraction=0.046)

imN = axN.imshow(piv_n.values, aspect='auto', cmap='Blues', origin='lower')
axN.set_xticks(range(len(piv_n.columns))); axN.set_xticklabels(piv_n.columns)
axN.set_yticks(range(len(piv_n.index)));   axN.set_yticklabels(piv_n.index)
axN.set_xlabel('div_gap'); axN.set_ylabel('close_in_range_min')
axN.set_title(f'n trades  (lookback={fixed_lb}, perp_thr={fixed_pt})')
for i in range(piv_n.shape[0]):
    for j in range(piv_n.shape[1]):
        v = piv_n.values[i, j]
        if pd.notna(v):
            axN.text(j, i, f'{int(v)}', ha='center', va='center',
                     color='black' if v < piv_n.values.max() * 0.5 else 'white',
                     fontsize=9)
plt.colorbar(imN, ax=axN, fraction=0.046)

plt.tight_layout(); plt.show()

### Best-by-each-lookback summary

What's the best combo at each lookback? Read this as: if you commit to a
sweep-window choice, here's the sweet spot for the other three thresholds.

In [ ]:
best_by_lb = (sweep[sweep['n'] >= 30]
              .sort_values('avg_R', ascending=False)
              .groupby('lookback')
              .head(1)
              .sort_values('lookback'))
best_by_lb.round(3)

## Walk-forward validation

The threshold sweep tested 500 combos. Even with the clustering pattern that
suggests a real plateau (not overfit spikes), the honest robustness check is
to evaluate on data the sweep didn't see.

Two checks:

1. **Conservative-pick OOS test.** Apply the chosen `SIGNAL_PARAMS`
   (lookback=24, div_gap=200, cir=0.10, perp_thr=400) to a held-out
   period it wasn't tuned on. If it still pays, the choice is robust.
2. **Train-optimal walk-forward.** For each fold, sweep on the train
   period to find the in-sample optimum, then evaluate that optimum on
   the next (out-of-sample) period. If train→test edge is preserved
   across multiple folds, the tuning process generalizes.

In [ ]:
# Conservative-pick on each period (no per-period tuning; just one set of
# params applied across all dates).
CONSERVATIVE = {'lookback': 24, 'div_gap': 200, 'cir': 0.10, 'perp_thr': 400}

def apply_params(precomputed_lb_df, dg, cir, pt) -> pd.DataFrame:
    f = precomputed_lb_df[(precomputed_lb_df['perp_cvd'] < -pt)
                          & (precomputed_lb_df['divergence'] > dg)
                          & (precomputed_lb_df['close_in_range'] >= cir)]
    if len(f) == 0:
        return f
    keep_ts = cooldown_select(f['trigger_ts'])
    return f[f['trigger_ts'].isin(keep_ts)]

def stats_block(df) -> dict:
    if len(df) == 0:
        return {'n': 0}
    wins   = df.loc[df['pnl_R'] > 0, 'pnl_R'].sum()
    losses = -df.loc[df['pnl_R'] < 0, 'pnl_R'].sum()
    pf = wins / losses if losses > 0 else float('inf')
    return {
        'n':       len(df),
        'win_pct': (df['pnl_R'] > 0).mean(),
        'avg_R':   df['pnl_R'].mean(),
        'sum_R':   df['pnl_R'].sum(),
        'PF':      pf,
    }

PERIODS = [
    ('2022',    pd.Timestamp('2022-01-30', tz='UTC'), pd.Timestamp('2023-01-01', tz='UTC')),
    ('2023',    pd.Timestamp('2023-01-01', tz='UTC'), pd.Timestamp('2024-01-01', tz='UTC')),
    ('2024',    pd.Timestamp('2024-01-01', tz='UTC'), pd.Timestamp('2025-01-01', tz='UTC')),
    ('2025',    pd.Timestamp('2025-01-01', tz='UTC'), pd.Timestamp('2026-01-01', tz='UTC')),
    ('2026YTD', pd.Timestamp('2026-01-01', tz='UTC'), pd.Timestamp('2027-01-01', tz='UTC')),
]

# Apply conservative pick across each calendar year
pre = precomputed[CONSERVATIVE['lookback']]
cons_rows = []
for name, t0, t1 in PERIODS:
    yr_df = pre[(pre['trigger_ts'] >= t0) & (pre['trigger_ts'] < t1)]
    filt = apply_params(yr_df, CONSERVATIVE['div_gap'], CONSERVATIVE['cir'], CONSERVATIVE['perp_thr'])
    s = stats_block(filt); s['period'] = name
    cons_rows.append(s)

cons_df = pd.DataFrame(cons_rows).set_index('period')
print('CONSERVATIVE PICK across all years (no per-period tuning):')
print(cons_df.round(3).to_string())

### Train-optimal walk-forward (3 folds, expanding window)

For each fold, sweep on the train set, pick the **best `avg_R` combo with
n ≥ 15 train trades**, then apply that combo to the next test period.
Compare train avg R vs test avg R per fold:

- Fold A: train 2022-2023, test 2024
- Fold B: train 2022-2024, test 2025
- Fold C: train 2022-2025, test 2026 YTD

In [ ]:
FOLDS = [
    ('A', '2022-2023', '2024',
        pd.Timestamp('2022-01-30', tz='UTC'), pd.Timestamp('2024-01-01', tz='UTC'),
        pd.Timestamp('2024-01-01', tz='UTC'), pd.Timestamp('2025-01-01', tz='UTC')),
    ('B', '2022-2024', '2025',
        pd.Timestamp('2022-01-30', tz='UTC'), pd.Timestamp('2025-01-01', tz='UTC'),
        pd.Timestamp('2025-01-01', tz='UTC'), pd.Timestamp('2026-01-01', tz='UTC')),
    ('C', '2022-2025', '2026YTD',
        pd.Timestamp('2022-01-30', tz='UTC'), pd.Timestamp('2026-01-01', tz='UTC'),
        pd.Timestamp('2026-01-01', tz='UTC'), pd.Timestamp('2027-01-01', tz='UTC')),
]

def best_combo_on_train(pre_train: pd.DataFrame, min_n: int = 15) -> dict | None:
    best = None
    for dg in DIV_GAPS:
        for cir in CIR_MINS:
            for pt in PERP_THRS:
                f = pre_train[(pre_train['perp_cvd'] < -pt)
                              & (pre_train['divergence'] > dg)
                              & (pre_train['close_in_range'] >= cir)]
                if len(f) == 0: continue
                keep_ts = cooldown_select(f['trigger_ts'])
                ff = f[f['trigger_ts'].isin(keep_ts)]
                if len(ff) < min_n: continue
                avg_R = ff['pnl_R'].mean()
                if best is None or avg_R > best['avg_R']:
                    best = {'div_gap': dg, 'cir': cir, 'perp_thr': pt,
                            'n_train': len(ff), 'avg_R': avg_R,
                            'sum_R': ff['pnl_R'].sum()}
    return best

# Walk-forward across all 4 lookbacks; pick the lookback whose train edge is
# most stable across folds rather than just the best in any single fold.
wf_rows = []
for lb in LOOKBACKS:
    pre = precomputed[lb]
    for fold_id, train_label, test_label, t0, t1, s0, s1 in FOLDS:
        pre_tr = pre[(pre['trigger_ts'] >= t0) & (pre['trigger_ts'] < t1)]
        pre_te = pre[(pre['trigger_ts'] >= s0) & (pre['trigger_ts'] < s1)]
        best = best_combo_on_train(pre_tr)
        if best is None:
            wf_rows.append({'lb': lb, 'fold': fold_id, 'train': train_label, 'test': test_label,
                            'n_train': 0, 'train_avg_R': None,
                            'n_test': 0, 'test_avg_R': None, 'verdict': 'no train sample'})
            continue
        test_f = apply_params(pre_te, best['div_gap'], best['cir'], best['perp_thr'])
        n_test = len(test_f)
        test_avg = test_f['pnl_R'].mean() if n_test > 0 else None
        test_sum = test_f['pnl_R'].sum() if n_test > 0 else 0
        wf_rows.append({
            'lb': lb, 'fold': fold_id, 'train': train_label, 'test': test_label,
            'best_dg': best['div_gap'], 'best_cir': best['cir'], 'best_pt': best['perp_thr'],
            'n_train': best['n_train'], 'train_avg_R': best['avg_R'],
            'n_test': n_test, 'test_avg_R': test_avg, 'test_sum_R': test_sum,
        })

wf = pd.DataFrame(wf_rows)
print('Train-optimal walk-forward:')
print(wf.round(3).to_string(index=False))

### Walk-forward verdict

Read the table above this way:

- **If `test_avg_R > 0` consistently across folds**, the threshold-tuning
  process generalizes — picking params on past data carries forward edge.
- **If `test_avg_R` is much smaller than `train_avg_R` and sometimes
  negative**, we're overfitting in the train sweep.
- **If `test_avg_R` matches `train_avg_R`** within roughly the standard
  error of the test sample, the plateau is real.

A reasonable rule of thumb: deploy only if the conservative-pick OOS years
(2025, 2026 YTD in the table above) show **positive avg_R and PF > 1.3**.

## Context features per trigger (log-only, no filtering)

For each historical trigger, attach the "bigger picture" features that
*weren't* in the trigger rule itself. Goal: spot whether any of them
separates winners from losers, without baking the filters in yet (sample
is small, filters are easy to overfit).

Features attached:
- **weekday, hour bin** — when within the week/day the trigger fired
- **fund_at_trigger** — funding rate at the trigger bar (live, not aggregated)
- **above_ema_7w** — BTC vs 7-week EMA (trend regime)
- **rv_pctile** — current 7-day realized-vol percentile vs history
- **lsr_pctile** — long/short ratio percentile at trigger date
- **fg_value** — Fear & Greed index at trigger date
- **near_event** — scheduled event (FOMC/CPI/etc.) within ±1 day
- **asia_oi_pct, asia_rng_pct** — daily macro strength

These are computed but not used by the live trigger. When we paper-track
forward signals, we'll log this same vector so we can compare the live
distribution to the historical one.

In [ ]:
import datetime as _dt

# Build per-trade frame from the best-TP long results computed earlier.
trades = results[('long', best_long)].copy()
trades['date']    = pd.to_datetime(trades['trigger_ts']).dt.date
trades['year']    = pd.to_datetime(trades['trigger_ts']).dt.year
trades['hour']    = pd.to_datetime(trades['trigger_ts']).dt.hour
trades['weekday'] = pd.to_datetime(trades['trigger_ts']).dt.weekday  # 0=Mon

def _hour_bin(h):
    if h < 10:  return '07-10 lon-early'
    if h < 14:  return '10-14 lon-late'
    if h < 17:  return '14-17 ny-early'
    return '17-21 ny-late'
trades['hour_bin'] = trades['hour'].map(_hour_bin)
trades['wday_name'] = trades['weekday'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})

# Funding at trigger (most recent funding settlement)
def _fund_at(ts):
    sl = fund_h['fr_close'].loc[:ts]
    return sl.iloc[-1] if len(sl) else np.nan
trades['fund_at_trigger'] = trades['trigger_ts'].map(_fund_at)

# Daily-context lookup from day_ctx
day_ctx_lookup = day_ctx[['oi_pct', 'fund_mean']].to_dict('index')
def _ctx(d, field):
    row = day_ctx_lookup.get(d)
    return row[field] if row is not None else np.nan
trades['asia_oi_pct'] = trades['date'].map(lambda d: _ctx(d, 'oi_pct'))

# BTC weekly trend regime — 7-week EMA on daily closes
spot_d = perp_h[['close']].resample('1D').last().dropna().copy()
spot_d['ema_7w']      = spot_d['close'].ewm(span=49, adjust=False).mean()
spot_d['above_ema']   = spot_d['close'] > spot_d['ema_7w']
def _above_ema(ts):
    d = pd.Timestamp(ts).normalize()
    sl = spot_d.loc[:d]
    return bool(sl['above_ema'].iloc[-1]) if len(sl) else None
trades['above_ema_7w'] = trades['trigger_ts'].map(_above_ema)

# 7-day realized vol percentile
spot_h_close = perp_h['close'].resample('1h').last().ffill()
log_ret = np.log(spot_h_close).diff()
rv_7d = (log_ret.rolling(24 * 7).std() * np.sqrt(24 * 365))
def _rv_pctile(ts):
    sl = rv_7d.loc[:ts]
    if len(sl) < 30: return np.nan
    return (sl <= sl.iloc[-1]).mean()
trades['rv_pctile'] = trades['trigger_ts'].map(_rv_pctile)

# Long/Short ratio percentile (BTC daily)
con = sqlite3.connect(str(DB))
lsr = pd.read_sql("SELECT timestamp, ratio FROM ca_long_short_ratio WHERE asset='BTC' ORDER BY timestamp", con)
con.close()
lsr['date'] = pd.to_datetime(lsr['timestamp'], unit='s', utc=True).dt.date
lsr_daily = lsr.set_index('date')['ratio']
lsr_pct = lsr_daily.expanding(min_periods=30).rank(pct=True)
trades['lsr_pctile'] = trades['date'].map(lsr_pct.to_dict())

# Fear & Greed
con = sqlite3.connect(str(DB))
fg = pd.read_sql("SELECT date, value FROM fear_greed_index", con)
con.close()
fg['date'] = pd.to_datetime(fg['date']).dt.date
fg_map = fg.set_index('date')['value'].to_dict()
trades['fg_value'] = trades['date'].map(fg_map)

# Scheduled event within +/- 1 day
con = sqlite3.connect(str(DB))
events = pd.read_sql("SELECT date, event_type FROM scheduled_events", con)
con.close()
events['date'] = pd.to_datetime(events['date']).dt.date
event_dates = set(events['date'])
def _near_event(d):
    return any((d + _dt.timedelta(days=k)) in event_dates for k in range(-1, 2))
trades['near_event'] = trades['date'].map(_near_event)

print(f'context-features per trade ({len(trades)} trades):')
display_cols = ['trigger_ts', 'pnl_R', 'wday_name', 'hour_bin', 'fund_at_trigger',
                'above_ema_7w', 'lsr_pctile', 'fg_value', 'near_event']
print(trades[display_cols].head(10).round(4).to_string(index=False))

### Bin-by-bin breakdown — which features separate winners from losers?

For each feature, group trades into buckets and show n / win% / avg_R.
This is *descriptive*, not prescriptive — filters could be picked from
here but each one further shrinks an already-tight sample.

In [ ]:
def show_bin(label, col_name):
    g = trades.groupby(col_name).agg(
        n=('pnl_R', 'size'),
        win_pct=('pnl_R', lambda x: (x > 0).mean()),
        avg_R=('pnl_R', 'mean'),
        sum_R=('pnl_R', 'sum'),
    ).round(3)
    print(f'\n--- {label} ---')
    print(g.to_string())

show_bin('weekday',                'wday_name')
show_bin('hour bin',               'hour_bin')
show_bin('BTC vs 7w EMA (regime)', 'above_ema_7w')
show_bin('near scheduled event',   'near_event')

def _fund_bin(f):
    if pd.isna(f): return 'unknown'
    bp = f * 1e4
    if bp < -0.5: return 'deep_neg <-0.5bp'
    if bp < 0:    return 'mild_neg -0.5..0'
    if bp < 0.5:  return 'near_zero 0..0.5'
    return 'positive >0.5'
trades['fund_bin'] = trades['fund_at_trigger'].map(_fund_bin)
show_bin('funding regime at trigger', 'fund_bin')

def _lsr_bin(p):
    if pd.isna(p): return 'unknown'
    if p < 0.33: return 'low (longs short)'
    if p < 0.67: return 'med'
    return 'high (longs long)'
trades['lsr_bin'] = trades['lsr_pctile'].map(_lsr_bin)
show_bin('Long/Short ratio percentile', 'lsr_bin')

def _fg_bin(v):
    if pd.isna(v): return 'unknown'
    if v < 25:  return 'extreme_fear'
    if v < 45:  return 'fear'
    if v < 55:  return 'neutral'
    if v < 75:  return 'greed'
    return 'extreme_greed'
trades['fg_bin'] = trades['fg_value'].map(_fg_bin)
show_bin('Fear & Greed regime', 'fg_bin')

### What this is for

Two reads:

1. **As of the historical 4-year sample**, the strongest signals are:
   - **Weekday**: Tue+Wed combined slightly negative (~-0.17R), other days
     strongly positive. Sample is small per bucket.
   - **LSR low percentile**: When most accounts are already short, the
     trade loses on average (~0.00R avg). Mechanism: the "everyone short"
     thesis is already priced in; the squeeze is finished before our entry.
   - **Hour bin**: NY late (17-21 UTC) is by far the strongest, London late
     (10-14 UTC) the weakest.
   - **Funding at trigger**: counter-intuitively, *positive* funding at
     trigger time wins more than deep-negative. Probably because by the
     time the bar fires, the squeeze has already started and funding has
     flipped.
2. **What we won't do yet**: bake any of these into a hard filter. Sample
   is too small. Instead: log these features on every live signal we
   paper-track. After 15-20 more triggers come in, compare the live
   distribution to the historical one — if a pattern holds, *then* add
   the filter.

Bar-level features (perp_cvd magnitude, divergence magnitude) **did not**
separate winners from losers in the t-test comparison. The CVD divergence
is a necessary trigger but doesn't predict outcome; context features carry
all the residual predictive information.

## Caveats & next steps

- **Sample size is tight.** With std-dev of R ≈ 1.3 over 60 trades, the
  95% CI on avg R is roughly `[mean - 0.33, mean + 0.33]`. A +0.25R mean
  has a lower CI bound around -0.08 — the edge is suggestive but not
  statistically nailed.
- **Regime dependence.** 2024 had only 5 trades (small sample) and lost.
  When short-macro days are rare in a year, this strategy is dormant.
  Worth running by era using [`r4_study/era_split.ipynb`](../r4_study/era_split.ipynb)
  boundaries.
- **Threshold sensitivity.** `SIGNAL_PARAMS` at the top of the trigger cell
  is uncalibrated — single-point thresholds picked by inspection. A grid
  sweep over divergence_gap × close_in_range × lookback would tell us how
  robust the edge is.
- **Spot-only execution.** `btc_1m` is spot only, but the trigger is perp.
  Spot/perp typically track within a few bp at 1m, so the stop/target
  walk should be close to right. For higher-fidelity execution, we'd need
  perp 1m bars too.
- **Spot CVD has more volume noise on quiet days.** At 15m, spot volume
  is often < 1k BTC; in those bars a "divergence" can be statistical
  noise. A volume floor on spot might tighten the signal.
- **Asymmetric edge.** Long side works, short side doesn't, despite the
  long-squeeze daily pattern being 4.6× more common. Worth understanding
  *why* before deploying.

If we ever deploy this, allocation should be tiny (~0.25% risk per trade,
~0.5-1% max NAV impact per signal) given the sample-size uncertainty.